# 2. Data Cleaning & Preprocessing

Notebook ini bertujuan untuk **membersihkan** dataset dari duplikat, missing value,
dan kolom yang tidak informatif.

> **Catatan penting:** Tidak ada scaling, SMOTE, atau feature selection berbasis target
> yang dilakukan di sini — semua itu dilakukan di dalam Pipeline saat training.

## 2.1 Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')

print('Library berhasil diimport.')

Library berhasil diimport.


## 2.2 Load Dataset Mentah

In [2]:
# Muat dataset dari file CSV asli
df = pd.read_csv('../dataset/agro_environmental_dataset.csv')
print(f'Dataset dimuat: {df.shape[0]:,} baris, {df.shape[1]} kolom')
df.head()

Dataset dimuat: 543,210 baris, 25 kolom


,location_id,soil_type,bulk_density,organic_matter_pct,cation_exchange_capacity,salinity_ec,buffering_capacity,soil_moisture_pct,moisture_limit_dry,moisture_limit_wet,...,soil_ph,ph_stress_flag,nitrogen_ppm,phosphorus_ppm,potassium_ppm,nutrient_balance,plant_category,suitability_score,stress_level,failure_flag
0,L00000,Clayey,1.1,4.0,30,0.5,0.9,5.17,25,52,...,6.48,0,100.1,50.8,121.3,excessive,vegetable,0.677,1,0
1,L00001,Alluvial,1.3,3.8,20,0.4,0.7,26.28,16,45,...,6.43,0,133.8,54.9,151.6,optimal,vegetable,0.871,0,0
2,L00002,Chalky,1.5,2.0,8,0.3,0.4,44.90,12,35,...,5.01,1,84.5,83.6,83.6,deficient,vegetable,0.000,2,1
3,L00003,Silty,1.4,3.0,18,0.4,0.6,27.05,18,42,...,5.41,1,168.2,30.5,220.0,deficient,cereal,0.510,1,0
4,L00004,Loamy,1.3,3.5,15,0.3,0.7,36.56,15,40,...,6.73,0,98.9,63.4,88.9,optimal,vegetable,1.000,0,0


## 2.3 Hapus Kolom Tidak Relevan

In [3]:
# location_id adalah ID unik per baris — tidak ada informasi prediktif
# Menghapusnya mencegah model 'hafal' ID alih-alih belajar pola

cols_to_drop = ['location_id']
df = df.drop(columns=cols_to_drop)

print(f'Kolom yang dihapus: {cols_to_drop}')
print(f'Ukuran setelah drop: {df.shape}')

Kolom yang dihapus: ['location_id']
Ukuran setelah drop: (543210, 24)


## 2.4 Cek & Hapus Duplikat

In [4]:
# Cek jumlah baris duplikat sebelum dihapus
n_before = len(df)
n_dup = df.duplicated().sum()
print(f'Baris duplikat ditemukan: {n_dup}')

# Hapus duplikat jika ada
if n_dup > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Duplikat dihapus. Baris berkurang: {n_before} → {len(df)}')
else:
    print('Tidak ada duplikat — dataset tetap.')

Baris duplikat ditemukan: 0
Tidak ada duplikat — dataset tetap.


## 2.5 Cek & Tangani Missing Value

In [5]:
# Ringkasan missing value per kolom
missing = df.isnull().sum()
missing_pct = missing / len(df) * 100

missing_df = pd.DataFrame({
    'Jumlah Missing': missing,
    'Persentase (%)': missing_pct.round(2)
})

has_missing = missing_df[missing_df['Jumlah Missing'] > 0]

if len(has_missing) > 0:
    print('=== Kolom dengan Missing Value ===')
    print(has_missing.to_string())
else:
    print('Tidak ada missing value pada dataset.')

Tidak ada missing value pada dataset.


In [6]:
# Tangani missing value jika ada
# Strategi:
#   - Numerik   → imputasi dengan median (lebih robust terhadap outlier)
#   - Kategorikal → imputasi dengan modus (nilai paling sering muncul)
# Catatan: imputasi dilakukan pada seluruh data di sini karena bersifat
# konstan (tidak bergantung target), sehingga tidak menyebabkan data leakage.

num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
# Gunakan hanya 'object' — pandas 3.x tidak mengizinkan 'str' di select_dtypes
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Imputasi numerik dengan median
for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'  [{col}] diisi dengan median = {median_val:.3f}')

# Imputasi kategorikal dengan modus
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f'  [{col}] diisi dengan modus = "{mode_val}"')

# Verifikasi akhir
remaining_missing = df.isnull().sum().sum()
print(f'\nTotal missing value tersisa: {remaining_missing}')


Total missing value tersisa: 0


## 2.6 Verifikasi Tipe Data

In [7]:
# Pastikan semua tipe data sudah sesuai
print('=== Tipe Data Setelah Cleaning ===')
print(df.dtypes)

=== Tipe Data Setelah Cleaning ===
soil_type                    object
bulk_density                float64
organic_matter_pct          float64
cation_exchange_capacity      int64
salinity_ec                 float64
buffering_capacity          float64
soil_moisture_pct           float64
moisture_limit_dry            int64
moisture_limit_wet            int64
moisture_regime              object
soil_temp_c                 float64
air_temp_c                  float64
thermal_regime               object
light_intensity_par         float64
soil_ph                     float64
ph_stress_flag                int64
nitrogen_ppm                float64
phosphorus_ppm              float64
potassium_ppm               float64
nutrient_balance             object
plant_category               object
suitability_score           float64
stress_level                  int64
failure_flag                  int64
dtype: object


In [8]:
# Identifikasi kolom numerik dan kategorikal (berguna sebagai acuan di notebook training)
num_features = [c for c in df.select_dtypes(include=['float64', 'int64']).columns
                if c != 'failure_flag']
# Gunakan hanya 'object' — pandas 3.x tidak mengizinkan 'str' di select_dtypes
cat_features = df.select_dtypes(include=['object']).columns.tolist()

print(f'Fitur Numerik ({len(num_features)}):')
print(' ', num_features)

print(f'\nFitur Kategorikal ({len(cat_features)}):')
print(' ', cat_features)

print(f'\nTarget: failure_flag')

Fitur Numerik (18):
  ['bulk_density', 'organic_matter_pct', 'cation_exchange_capacity', 'salinity_ec', 'buffering_capacity', 'soil_moisture_pct', 'moisture_limit_dry', 'moisture_limit_wet', 'soil_temp_c', 'air_temp_c', 'light_intensity_par', 'soil_ph', 'ph_stress_flag', 'nitrogen_ppm', 'phosphorus_ppm', 'potassium_ppm', 'suitability_score', 'stress_level']

Fitur Kategorikal (5):
  ['soil_type', 'moisture_regime', 'thermal_regime', 'nutrient_balance', 'plant_category']

Target: failure_flag


## 2.7 Ringkasan Akhir Dataset Bersih

In [9]:
print('=' * 55)
print('RINGKASAN DATASET SETELAH CLEANING')
print('=' * 55)
print(f'  Ukuran dataset   : {df.shape[0]:,} baris x {df.shape[1]} kolom')
print(f'  Missing value    : {df.isnull().sum().sum()}')
print(f'  Duplikat         : {df.duplicated().sum()}')
print(f'  Fitur numerik    : {len(num_features)}')
print(f'  Fitur kategorikal: {len(cat_features)}')
print(f'  Target (0/1)     : {(df["failure_flag"]==0).sum():,} / {(df["failure_flag"]==1).sum():,}')

df.head()

RINGKASAN DATASET SETELAH CLEANING
  Ukuran dataset   : 543,210 baris x 24 kolom
  Missing value    : 0
  Duplikat         : 0
  Fitur numerik    : 18
  Fitur kategorikal: 5
  Target (0/1)     : 455,130 / 88,080


,soil_type,bulk_density,organic_matter_pct,cation_exchange_capacity,salinity_ec,buffering_capacity,soil_moisture_pct,moisture_limit_dry,moisture_limit_wet,moisture_regime,...,soil_ph,ph_stress_flag,nitrogen_ppm,phosphorus_ppm,potassium_ppm,nutrient_balance,plant_category,suitability_score,stress_level,failure_flag
0,Clayey,1.1,4.0,30,0.5,0.9,5.17,25,52,dry,...,6.48,0,100.1,50.8,121.3,excessive,vegetable,0.677,1,0
1,Alluvial,1.3,3.8,20,0.4,0.7,26.28,16,45,optimal,...,6.43,0,133.8,54.9,151.6,optimal,vegetable,0.871,0,0
2,Chalky,1.5,2.0,8,0.3,0.4,44.90,12,35,waterlogged,...,5.01,1,84.5,83.6,83.6,deficient,vegetable,0.000,2,1
3,Silty,1.4,3.0,18,0.4,0.6,27.05,18,42,optimal,...,5.41,1,168.2,30.5,220.0,deficient,cereal,0.510,1,0
4,Loamy,1.3,3.5,15,0.3,0.7,36.56,15,40,optimal,...,6.73,0,98.9,63.4,88.9,optimal,vegetable,1.000,0,0


## 2.8 Simpan Dataset Bersih ke cleaning.csv

In [10]:
# Simpan hasil cleaning ke file CSV — tanpa index agar bersih
output_path = '../dataset/cleaning.csv'
df.to_csv(output_path, index=False)

print(f'Dataset bersih berhasil disimpan ke: {output_path}')
print(f'Ukuran file: {df.shape[0]:,} baris x {df.shape[1]} kolom')

Dataset bersih berhasil disimpan ke: ../dataset/cleaning.csv
Ukuran file: 543,210 baris x 24 kolom
